<center
style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#006400; border-radius:20px; font-size:360%; text-align:center;padding:3.0px; background: #ffffff00; border-top: 13px solid #006400; border-left: 10px solid #006400; border-right: -5px solid #17202a">  Predicting Optimal Fertilizers <img src="https://www.kaggle.com/competitions/91717/images/header" width="280" style="border: 1px solid #17202a; border-radius: 1px" loading="lazy" />
</center>

# Load the tools and datasets 🎢

In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from xgboost import XGBClassifier, plot_importance, cv
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import top_k_accuracy_score
import warnings
warnings.filterwarnings('ignore')

include_ext = True

## Get the datasets 

In [2]:
# The target
target = 'Fertilizer Name'
# Load the training set
X = pd.read_csv('/kaggle/input/playground-series-s5e6/train.csv', index_col='id')
# Load external data
X_ext = pd.read_csv('/kaggle/input/d/irakozekelly/fertilizer-prediction/Fertilizer Prediction.csv')
X_ext.columns = X.columns

# Load the testing set
test_data = pd.read_csv('/kaggle/input/playground-series-s5e6/test.csv', index_col='id')

# Decide if external data should be included
if include_ext:
    X = pd.concat([X, X_ext], ignore_index=True)
    y = X.pop(target)
else:
    X = X
# Get the ext_target
y_ext = X_ext.pop(target)

In [3]:
X.tail(4)

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
849996,35,72,47,Loamy,Millets,38,1,32
849997,28,50,61,Sandy,Maize,10,11,14
849998,29,57,63,Loamy,Ground Nuts,7,10,4
849999,25,72,42,Sandy,Wheat,38,2,6


In [4]:
test_data.head(4)

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
id,,,,,,,,
750000,31,70,52,Sandy,Wheat,34,11,24
750001,27,62,45,Red,Sugarcane,30,14,15
750002,28,72,28,Clayey,Ground Nuts,14,15,4
750003,37,53,57,Black,Ground Nuts,18,17,36


## Define the metrics

In [5]:
# Get top-k predictions
def get_top_k_predictions(probs, k):
    return np.argsort(probs, axis=1)[:, -k:][:, ::-1]


# Single-label MAP@K
def mapk_single_label(y_true, y_pred, k=3):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)[:, :k]
    matches = (y_true.reshape(-1, 1) == y_pred)
    ranks = np.where(matches.any(axis=1), matches.argmax(axis=1) + 1, np.inf)
    return np.mean(ranks ** -1)

# Multi-label MAP@K (each instance has one label in a list)
def apk(actual, predicted, k=10):
    if not actual:
        return 0.0
    predicted = predicted[:k]
    score = 0.0
    num_hits = 0
    seen = set()
    actual_set = set(actual)
    for i, p in enumerate(predicted):
        if p in actual_set and p not in seen:
            num_hits += 1
            score += num_hits / (i + 1)
            seen.add(p)
    return score / min(len(actual), k)

def mapk(actual, predicted, k=10):
    return np.mean([apk([a], p, k) for a, p in zip(actual, predicted)])

# Prepare the data and targets 🎏
## Encode the target and cat_features 🕹️

In [6]:
# Encode labels if necessary
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# Covert num_features from int64 to lower memory int8
for col in X.select_dtypes('int64').columns:
    # train data
    X[col] = X[col].astype('int8')
    # external data
    X_ext[col] = X_ext[col].astype('int8')
    # test data
    test_data[col] = test_data[col].astype('int8')
    
# Encode the cat_features   
for cat_feat in ['Soil Type', 'Crop Type']:
    cat_le = LabelEncoder()
    # train data
    X[cat_feat] = cat_le.fit_transform(X[cat_feat])
    X[cat_feat] = X[cat_feat].astype('category')
    # external data
    X_ext[cat_feat] = cat_le.fit_transform(X_ext[cat_feat])
    X_ext[cat_feat] = X_ext[cat_feat].astype('category')
    # test data
    test_data[cat_feat] = cat_le.transform(test_data[cat_feat])
    test_data[cat_feat] = test_data[cat_feat].astype('category')

In [7]:
X['Temp_Humidity'] = X['Temparature'] * X['Humidity']
X['Temp_Moisture'] = X['Temparature'] * X['Moisture']
X['Humidity_Moisture'] = X['Humidity'] * X['Moisture']

X_ext['Temp_Humidity'] = X_ext['Temparature'] * X_ext['Humidity']
X_ext['Temp_Moisture'] = X_ext['Temparature'] * X_ext['Moisture']
X_ext['Humidity_Moisture'] = X_ext['Humidity'] * X_ext['Moisture']

test_data['Temp_Humidity'] = test_data['Temparature'] * test_data['Humidity']
test_data['Temp_Moisture'] = test_data['Temparature'] * test_data['Moisture']
test_data['Humidity_Moisture'] = test_data['Humidity'] * test_data['Moisture']

## Train Test Split ⚔️

In [8]:
# Split into train and test sets
X_train, X_valid, y_train, y_valid = train_test_split(X, y_encoded,
                                                      test_size=0.3,
                                                      random_state=4)

# Modeling 🎯
## Define the model

In [9]:
# Define the classifier
xgb_best_params = {
   'n_estimators': 5000,
    'max_depth':12,
    'subsample': 0.7,
    'colsample_bytree':0.5,
    'learning_rate':0.025, 
    'gamma':0.4,
    'max_delta_step': 5, # read more on 
    'early_stopping_rounds':200,
    'objective':'multi:softprob',
    'enable_categorical':True,
    'tree_method':'hist',
    'device':'cuda',
    'reg_alpha':2.7,
    'reg_lambda':1.6,
    'num_class':7,
    'num_parallel_tree': 5,
    'enable_categorical': True
}

## Cross Validation: oof

In [ ]:
my_spliter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for f, (tr_ind, va_ind) in enumerate(my_spliter.split(X, y_encoded), 1):
    clf = XGBClassifier(**xgb_best_params)
    X_tr, X_va = X.iloc[tr_ind], X.iloc[va_ind]
    y_tr, y_va = y_encoded[tr_ind], y_encoded[va_ind]
    print(f'\n🧮 Fitting Fold_{f}')
    clf.fit(X_tr, y_tr, 
            eval_set=[(X_va, y_va)],
            verbose=200)

    y_probs = clf.predict_proba(X_va)
    top_3_preds = get_top_k_predictions(y_probs, 3)
    map3 = mapk_single_label(y_va, top_3_preds, 3)
    print('\n🎯⚖️ map3: {:.5}'.format(map3))


🧮 Fitting Fold_1
[0]	validation_0-mlogloss:1.94566
[200]	validation_0-mlogloss:1.92182
[400]	validation_0-mlogloss:1.91347
[600]	validation_0-mlogloss:1.90938
[800]	validation_0-mlogloss:1.90763
[1000]	validation_0-mlogloss:1.90717
[1184]	validation_0-mlogloss:1.90752

🎯⚖️ map3: 0.34682

🧮 Fitting Fold_2
[0]	validation_0-mlogloss:1.94566
[200]	validation_0-mlogloss:1.92205
[400]	validation_0-mlogloss:1.91396
[600]	validation_0-mlogloss:1.91008
[800]	validation_0-mlogloss:1.90848
[1000]	validation_0-mlogloss:1.90817


## Fit the final model

In [ ]:
clf_final = XGBClassifier(**xgb_best_params)

# Train a classifier
clf_final.fit(X_train, y_train, 
              eval_set=[(X_valid, y_valid)],
              verbose=200)

## Evaluate the model on the validation data

In [ ]:
# Predict probabilities
y_probs = clf_final.predict_proba(X_valid)
y_probs[:3]

## Let's score over the length of the number of targets

In [ ]:
# Evaluate MAP@K for k = 1 to k_max
k_max = y.nunique()
k_values = range(1, k_max)
mapk_scores = []

for k in k_values:
    top_k_preds = get_top_k_predictions(y_probs, k)
    mapk_scores.append(mapk(y_valid, top_k_preds, k))

In [ ]:
# Plot the results
plt.figure(figsize=(8, 5))
ax = plt.plot(k_values, mapk_scores, marker='s', label='Multi-label MAP@K', linestyle='--')
plt.title(f'MAP@k for k in range 1 to {k_max-1}')
plt.xlabel('K')
plt.ylabel('MAP@K Score')
plt.xticks(k_values)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



# Prediction on the test set 🎰

In [ ]:
test_proba = clf.predict_proba(test_data)
test_proba[:1]

## Sort the predicted probabilities

In [ ]:
preds = np.argsort(test_proba, axis=1)[:, ::-1]
preds[:10]

## Pick the top three

In [ ]:
test_top_3 = np.argsort(test_proba, axis=1)[:, -3:][:, ::-1]
test_top_3

## Decode the top three picks

In [ ]:
test_top_3_names = target_encoder.inverse_transform(test_top_3.ravel())
test_3_picks = test_top_3_names.reshape(test_top_3.shape)

test_3_picks

In [ ]:
picks = pd.DataFrame(test_3_picks, columns=['First', 'Second', 'Thirth']).apply(lambda x: x)
picks

In [ ]:
colors = ['grey', 'red', 'blue', 'orange']

plt.figure(figsize=(12,3))
for n, pick_rank in enumerate(picks.columns, start=1):
    plt.subplot(1, 3, n)
    sns.countplot(picks.sort_values(by=pick_rank), x=pick_rank, dodge=False)
    plt.xticks(rotation=90)
    plt.title(f'Count as {pick_rank} pick')
    if n!=1:
        plt.yticks([])
    plt.ylabel('')
plt.show()

# Prepare the submission file 🥂

<center>
<img src="https://eos.com/wp-content/uploads/2023/11/components-of-different-types-of-fertilizers.jpg" width="500"/>
</center>

In [ ]:
# prep the submission dataframe
preds_df = pd.DataFrame({
    'id': test_data.index,
    'Fertilizer Name': [' '.join(preds) for preds in test_3_picks]
})

preds_df.head(10)

In [ ]:
preds_df.to_csv('submission.csv', index=False)
print("Let's submit to the competition.")

# Appendix

## 👀 How are the predicted probabilities from first to last distributed

In [ ]:
# Create the figure and GridSpec layout
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, width_ratios=[3, 2])

# Decode the classes
target_classes = target_encoder.classes_

# First plot: Heatmap spanning both columns
ax0 = fig.add_subplot(gs[:, 0])
sns.heatmap(test_proba, cmap='copper_r', ax=ax0)
ax0.set_xticks(ticks=np.arange(7) + 0.5)
ax0.set_xticklabels(target_classes, rotation=-45)
ax0.set_title('Predicted probabilities across the test set', color='maroon')

# Second plot: Sorted probabilities
ax1 = fig.add_subplot(gs[0, 1])
for i in range(1, 7):
    pd.Series(np.sort(test_proba, axis=1)[:, i]).plot(ax=ax1)
ax1.set_title('Sorted predicted probabilities for test set', color='maroon')

# Third plot: Max, Min, Mean probabilities
ax2 = fig.add_subplot(gs[1, 1])
pd.Series(np.max(test_proba, axis=1)).plot(ax=ax2, label='Max')
pd.Series(np.min(test_proba, axis=1)).plot(ax=ax2, label='Min')
pd.Series(np.mean(test_proba, axis=1)).plot(ax=ax2, color='darkgreen', label='Mean')
ax2.set_title('Scope of Max, Min, Mean probabilities', color='maroon')
ax2.legend()

plt.tight_layout()
plt.show()